In [ ]:
import pandas as pd
import numpy as np
from functools import reduce
from scipy.stats import ttest_ind
import matplotlib.pyplot as plt
import matplotlib as mpl
from matplotlib import font_manager
from matplotlib.colors import LinearSegmentedColormap

arial_path = "/media/scratch/fy2306/tools/fonts"
font_files = font_manager.findSystemFonts(fontpaths=arial_path)

for file in font_files:
    font_manager.fontManager.addfont(file)
    
mpl.rcParams['font.family'] = 'Arial'
import seaborn as sns
import warnings
warnings.filterwarnings("ignore")

In [ ]:
# from grna_phenotype_score
def pam_lfc_heatmap(
		myc_var,
		date,
		test_name,
		baseline,
		treatments,
		day_numbers,
		mode,
		per_time_unit=True):
	
	# merge 4 replicates per sgRNA
	df_lfc = None
	for treatment in treatments:
		df_lfc_path = f"/media/scratch/fy2306/projects/base_editing/data/{date}/mageck/{test_name}/MYC_U1.{treatment}_vs_{baseline}.sgrna_summary.txt"
		df_lfc_rep = pd.read_csv(df_lfc_path, sep="\t", usecols=["sgrna", "LFC"])
		df_lfc_rep = df_lfc_rep.rename(columns={"LFC": f"per_day_LFC_{treatment}"})
		if df_lfc is None:
			df_lfc = df_lfc_rep
		else:
			# Merge on 'sgrna'
			df_lfc = pd.merge(df_lfc, df_lfc_rep, on="sgrna", how="inner")
	print(len(df_lfc))

	treatment_lfc_columns = [f"per_day_LFC_{treatment}" for treatment in treatments]

	if per_time_unit:
		for col, day in zip(treatment_lfc_columns, day_numbers):
			df_lfc[col] = df_lfc[col] / (day + 0) # per time unit.

	lib_path = f"/media/scratch/fy2306/projects/base_editing/data/grna_type/MYC-lib-for-mageck.type.{myc_var}.organized.txt"
	df_lib = pd.read_csv(lib_path, sep="\t", usecols=["id", "grna_target_seq", "grna_target_seq_rc", "seq_name", "grna_target_seq_strand", "pam", "seq_chr", "indel_base_pos_chr", "indel_consequences", "indel_repeats"])

	df = pd.merge(df_lfc, df_lib, left_on="sgrna", right_on="id", how="right")
	print(len(df_lfc))
	print(len(df))

	df = df[(df['seq_name'].isin(["MYC_GFP", "MYC_SNP1", "MYC_SNP2", "MYC_SNP3", "MYC_STOP"])) & (df['pam'] != "startORend") & (df['pam'] != "non-targeting") & (df['indel_consequences'] != "GFP") & (df['indel_repeats'] == False)]
	print(len(df))
	
	if mode == "NXX":
		df['pam'] = df['pam'].str.replace(r'^.', 'N', regex=True)	# NXX
	elif mode == "NXN":
		df['pam'] = df['pam'].str.replace(r'^.|.$', 'N', regex=True)	# NXN

	g = (
		df.groupby('pam', dropna=False)[treatment_lfc_columns]
		.mean(numeric_only=True)
	)

	g["sgRNA_phenotype_score"] = g.mean(axis=1, skipna=True)
	g = g.assign(pam=g.index)


	df = g.copy()
	df["y"] = df["pam"].str[1]   # 2nd letter
	df["x"] = df["pam"].str[2]   # 3rd letter

	M = df.pivot(index="y", columns="x", values="sgRNA_phenotype_score")

	order = ["A", "T", "C", "G"]
	M = M.reindex(index=order, columns=order)

	cmap = LinearSegmentedColormap.from_list("AA66FF", ["#6B429D", "#EADCFC"])

	fig, ax = plt.subplots(figsize=(4,5))
	im = ax.imshow(M.values, cmap=cmap, aspect="auto", origin="lower")

	for i, y in enumerate(M.index):
		for j, x in enumerate(M.columns):
			pam = f"N{x}{y}"
			ax.text(j, i, pam, ha="center", va="center", fontsize=13, color="white")


	ax.set_xticks(range(M.shape[1]))
	ax.set_xticklabels(M.columns)
	ax.set_yticks(range(M.shape[0]))
	ax.set_yticklabels(M.index)

	ax.tick_params(axis="x", labelsize=14)
	ax.tick_params(axis="y", labelsize=14)
	ax.set_xlabel("3rd base in PAM", fontsize=14)
	ax.set_ylabel("2nd base in PAM", fontsize=14)

	cbar = plt.colorbar(im, ax=ax, orientation='horizontal', pad=0.15)     
	cbar.set_ticks([-0.016, -0.012, -0.008, -0.004])
	cbar.ax.tick_params(labelsize=13) 
	cbar.set_label(label="Average sgRNA phenotype score",fontsize=14)    
	for s in ax.spines.values():
		s.set_visible(False)
	for s in cbar.ax.spines.values():
		s.set_visible(False)
	plt.tight_layout()
	plt.savefig(f"/media/scratch/fy2306/projects/base_editing/plots/screening/sgRNA_lfc_pam_heatmap.{mode}.pdf", 
				bbox_inches="tight",
				dpi=300,              
				transparent=True,
				format='pdf')
	plt.show()

In [ ]:
myc_var = "myc2_indel"
date = "20250416"
test_name = "test-1-standard/test"
baseline = "Cas9-HMOI_D0"
treatments = ["Cas9-LMOI_D20", "Cas9-LMOI_D8", "Cas9_SpRY_L_MOI_D_20", "Cas9_SpRY_L_MOI_D_8"]
day_numbers = [20, 8, 20, 8]
merge_method_replicate = "mean"
pam_lfc_heatmap(
		myc_var,
		date,
		test_name,
		baseline,
		treatments,
		day_numbers,
		mode="NXX",
		per_time_unit=True)

In [ ]:
def pam_top_sgrna(
		myc_var, date, 
		test_name, 
		baseline, 
		treatments, 
		day_numbers, 
		merge_method_replicate, 
		top_percent=0.1,
		method='abs',
		mode="NXN",
		per_time_unit=True):
	
	# merge 4 replicates
	# copied from handling negative control sgRNAs
	df_test = None
	for treatment in treatments:
		df_test_path = f"/media/scratch/fy2306/projects/base_editing/data/{date}/mageck/{test_name}/MYC_U1.{treatment}_vs_{baseline}.sgrna_summary.txt"
		df_test_rep = pd.read_csv(df_test_path, sep="\t", usecols=["sgrna", "Gene", "LFC"])
		df_test_rep = df_test_rep[df_test_rep["Gene"].isin(["MYC_GFP", "MYC_SNP1", "MYC_SNP2", "MYC_SNP3", "MYC_STOP"])]
		df_test_rep = df_test_rep.drop('Gene', axis=1)
		df_test_rep = df_test_rep.rename(columns={"LFC": f"LFC_{treatment}"})
		if df_test is None:
			df_test = df_test_rep
		else:
			# Merge on 'sgrna'
			df_test = pd.merge(df_test, df_test_rep, on="sgrna", how="inner")
	print(len(df_test))

	treatment_lfc_columns = [f"LFC_{treatment}" for treatment in treatments]

	if per_time_unit:
		for col, day in zip(treatment_lfc_columns, day_numbers):
			df_test[col] = df_test[col] / (day + 0) # per time unit. 

	if merge_method_replicate == "mean":
		df_test["per time unit LFC"] = df_test[treatment_lfc_columns].mean(axis=1)
	df_test = df_test[["sgrna", "per time unit LFC"]]

	df_lib_path = f"/media/scratch/fy2306/projects/base_editing/data/grna_type/MYC-lib-for-mageck.type.{myc_var}.organized.txt"
	df_lib = pd.read_csv(df_lib_path, sep="\t", usecols=["id", "seq_name", "pam", "indel_consequences", "indel_repeats"])

	df = pd.merge(df_lib, df_test, left_on='id', right_on='sgrna')

	df = df[
		(df['seq_name'].isin(["MYC_GFP", "MYC_SNP1", "MYC_SNP2", "MYC_SNP3", "MYC_STOP"])) & 
		(df['pam'] != "startORend") & 
		(df['pam'] != "non-targeting") & 
		(df['indel_consequences'] != "GFP") & 
		(df['indel_repeats'] == False)]
	
	if mode == "NXX":
		df['pam'] = df['pam'].str.replace(r'^.', 'N', regex=True)	# NXX
		selected_pam = "NGG"
	elif mode == "NXN":
		df['pam'] = df['pam'].str.replace(r'^.|.$', 'N', regex=True)	# NXN
		selected_pam = "NGN"
	
	if method == 'abs':
		df['rank_score'] = df['per time unit LFC'].abs()
	elif method == 'pos':
		df['rank_score'] = df['per time unit LFC']
	elif method == 'neg':
		df['rank_score'] = -df['per time unit LFC']

	T = int(len(df) * top_percent)
	top_df = df.nlargest(T, 'rank_score')

	pam_counts = top_df['pam'].value_counts()

	if selected_pam in pam_counts.index:
		others = pam_counts.drop(selected_pam)
		pam_counts = pd.concat([pam_counts.loc[[selected_pam]], others])
	else:
		pam_counts = pam_counts.sort_values(ascending=False)

	labels = pam_counts.index.tolist()
	counts = pam_counts.values.tolist()
	n_other = sum(lab != selected_pam for lab in labels)
	cmap = plt.cm.get_cmap("gist_earth", n_other + 2)
	blue_levels = [cmap(i+1) for i in range(n_other)]

	colors = []
	j = 0
	for lab in labels:
		if lab == selected_pam:
			colors.append("#FF0066")
		else:
			colors.append(blue_levels[j])
			j += 1

	def autopct_if_big(pct):
		return f"{pct:.1f}%" if pct >= 4 else ""
	
	plt.figure(figsize=(5, 5))
	wedges, texts, autotexts = plt.pie(
		counts,
		labels=labels,
		autopct=autopct_if_big,
		startangle=90,
		colors=colors,
		labeldistance=1.075,
		pctdistance=0.65,
		textprops={'fontsize': 11, 'color': 'black'}
	)

	# plt.legend(
	# 	wedges,
	# 	labels,
	# 	loc='lower center',
	# 	bbox_to_anchor=(0.5, -0.15),
	# 	ncol=4,
	# 	frameon=False,
	# 	fontsize=11
	# )

	for t in autotexts:
		t.set_color("white")
		t.set_fontsize(11)

	plt.title(f'top {top_percent*100:.0f}% sgRNAs by {method}', fontsize=14)
	plt.tight_layout()
	plt.savefig(f"/media/scratch/fy2306/projects/base_editing/plots/screening/sgRNA_lfc_pam_toppct.{mode}.pdf", 
				bbox_inches="tight",
				dpi=300,              
				transparent=True,
				format='pdf')
	plt.show()
	

In [ ]:
myc_var = "myc2_indel"
date = "20250416"
test_name = "test-1-standard/test"
baseline = "Cas9-HMOI_D0"
treatments = ["Cas9-LMOI_D20", "Cas9-LMOI_D8", "Cas9_SpRY_L_MOI_D_20", "Cas9_SpRY_L_MOI_D_8"]
day_numbers = [20, 8, 20, 8]
merge_method_replicate = "mean"
pam_top_sgrna(
		myc_var, date, 
		test_name, 
		baseline, 
		treatments, 
		day_numbers, 
		merge_method_replicate, 
		top_percent=0.05,
		method='abs',
		mode="NXX",
		per_time_unit=True)